<a href="https://colab.research.google.com/github/gitjohnpaul/-FUNDAI-Laboratories-NELLAS/blob/main/Lab5_Rule_Based_Expert_System_Nellas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 5: Design a Rule-Based Expert System

## Fundamentals of Artificial Intelligence

**Name:** John Paul S. Nellas

**Course:** BSCS-AI

**Section:** 09282-FUNDAI

**Date:** September 16, 2026

**Domain:** Medical Symptom Triage Bot

**GitHub URL:** https://github.com/gitjohnpaul/-FUNDAI-Laboratories-NELLAS

## Disclaimer
This expert system is for educational purposes only. It is not a real medical diagnostic system.

In [1]:
RULES = [
    {
        "id": "R1",
        "priority": 10,
        "if": {
            "fever": True,
            "cough": True,
            "body_ache": True
        },
        "then": {
            "flu_like": True
        },
        "description": "Fever, cough, and body ache suggest flu-like symptoms."
    },

    {
        "id": "R2",
        "priority": 9,
        "if": {
            "fever": True,
            "sore_throat": True
        },
        "then": {
            "throat_infection_possible": True
        },
        "description": "Fever and sore throat suggest possible throat infection."
    },

    {
        "id": "R3",
        "priority": 9,
        "if": {
            "cough": True,
            "shortness_of_breath": True
        },
        "then": {
            "respiratory_warning": True
        },
        "description": "Cough and shortness of breath require medical attention."
    },

    {
        "id": "R4",
        "priority": 8,
        "if": {
            "flu_like": True
        },
        "then": {
            "suspect_flu": True
        },
        "description": "Flu-like symptoms lead to suspected flu."
    },

    {
        "id": "R5",
        "priority": 7,
        "if": {
            "suspect_flu": True
        },
        "then": {
            "recommend_rest_and_hydration": True
        },
        "description": "Suspected flu recommends rest and hydration."
    },

    {
        "id": "R6",
        "priority": 10,
        "if": {
            "respiratory_warning": True
        },
        "then": {
            "recommend_doctor": True
        },
        "description": "Respiratory warning recommends doctor consultation."
    },

    {
        "id": "R7",
        "priority": 12,
        "if": {
            "severe_chest_pain": True
        },
        "then": {
            "emergency": True
        },
        "description": "Severe chest pain requires emergency care."
    },

    {
        "id": "R8",
        "priority": 6,
        "if": {
            "throat_infection_possible": True
        },
        "then": {
            "recommend_doctor": True
        },
        "description": "Possbile throat infection recommends doctor consultation."
    }
]

In [2]:
DEFAULT_FACTS = {
    "fever": False,
    "cough": False,
    "body_ache": False,
    "sore_throat": False,
    "shortness_of_breath": False,
    "severe_chest_pain": False
}

In [3]:
def make_facts(**overrides):
    facts = dict(DEFAULT_FACTS)
    facts.update(overrides)
    return facts

In [4]:
def rule_condition_matches(rule_if, facts):
    for fact_name, required_value in rule_if.items():
        if facts.get(fact_name, False) != required_value:
            return False
    return True

In [23]:
def forward_chain(facts, rules, trace=True):
    facts = dict(facts)
    fired_rules = []
    sorted_rules = sorted(rules, key=lambda rule: rule.get("priority", 0), reverse=True)
    changed = True

    while changed:
        changed = False
        for rule in sorted_rules:
            if rule["id"] in fired_rules:
                continue
            if rule_condition_matches(rule["if"], facts):
                if trace:
                    print(f"Rule {rule['id']} fired.")
                    print(f"Description: {rule.get('description', 'No description')}")
                    print(f"New facts added: {rule['then']}")
                    print("-" * 50)
                for fact_name, fact_value in rule["then"].items():
                    if facts.get(fact_name) != fact_value:
                        facts[fact_name] = fact_value
                        changed = True
                fired_rules.append(rule["id"])

    return facts, fired_rules   # <- now outside the while loop

In [6]:
def explain_inference(fired_ruled_ids, rules):
    rule_map = {rule["id"]: rule for rule in rules}

    print("=" * 60)
    print("EXPLANATION OF INFERENCE")
    print("=" * 60)

    if len(fired_ruled_ids) == 0:
        print("No rules were fired.")
        return

    for rule_id in fired_ruled_ids:
        rule = rule_map[rule_id]

        print(f"Rule {rule_id}")
        print(f"IF: {rule['if']}")
        print(f"THEN: {rule['then']}")
        print(f"Reason: {rule.get('description', 'No description')}")
        print("=" * 60)

In [7]:
ADVICE_PRIORITY = [
    (
        "emergency",
        "EMERGENCY: Please seek immediate medical help."
    ),

    (
        "recommend_doctor",
        "RECOMMENDATION: Please consult a doctor."
    ),

    (
        "suspect_flu",
        "POSSIBLE CONDITION: Flu-like illness."
    ),

    (
        "throat_infection_possible",
        "POSSIBLE CONDITION: Possible throat infection."
    ),

    (
        "respiratory_warning",
        "WARNING: Respiratory symptoms detected."
    ),

    (
        "recommend_rest_and_hydration",
        "ADVICE: Rest and stay hydrated."
    )

]

def get_conclusions(facts):
    conclusions = []

    for fact_name, message in ADVICE_PRIORITY:
        if facts.get(fact_name, False):
            conclusions.append(message)

    if len(conclusions) == 0:
        conclusions. append(
            "No specific condition matched."
            "Please provide more symptoms or consult a professional."
        )

    return conclusions

In [8]:
def run_case(case_name, facts):
    print("=" * 60)
    print(f"CASE: {case_name}")
    print("=" * 60)
    print("Input facts:")
    print(facts)
    print("-" * 60)

    final_facts, fired_rules = forward_chain(facts, RULES, trace=True)

    print("Final facts:")
    print(final_facts)

    print("\nConclusions:")
    for conclusion in get_conclusions(final_facts):
        print("-", conclusion)

    print()

    explain_inference(fired_rules, RULES)

    print("\n\n")

In [15]:
# Case 1: Flu-like symptoms
case1 = make_facts(
    fever=True,
    cough=True,
    body_ache=True
)

# Case 2: Respiratory warning
case2 = make_facts(
    cough=True,
    shortness_of_breath=True
)

# Case 3: Emergency condition
case3 = make_facts(
    severe_chest_pain=True
)

# Case 4: No matching symptoms
case4 = make_facts()

run_case("Flu-like symptoms", case1)
run_case("Respiratory warning", case2)
run_case("Emergency chest pain", case3)
run_case("No symptoms", case4)

CASE: Flu-like symptoms
Input facts:
{'fever': True, 'cough': True, 'body_ache': True, 'sore_throat': False, 'shortness_of_breath': False, 'severe_chest_pain': False}
------------------------------------------------------------
Rule R1 fired.
Description: Fever, cough, and body ache suggest flu-like symptoms.
New facts added: {'flu_like': True}
--------------------------------------------------
Rule R4 fired.
Description: Flu-like symptoms lead to suspected flu.
New facts added: {'suspect_flu': True}
--------------------------------------------------
Rule R5 fired.
Description: Suspected flu recommends rest and hydration.
New facts added: {'recommend_rest_and_hydration': True}
--------------------------------------------------
Final facts:
{'fever': True, 'cough': True, 'body_ache': True, 'sore_throat': False, 'shortness_of_breath': False, 'severe_chest_pain': False, 'flu_like': True, 'suspect_flu': True, 'recommend_rest_and_hydration': True}

Conclusions:
- POSSIBLE CONDITION: Flu-lik

In [17]:

def ask_yes_no(question):
    answer = input(question + " (yes/no): ")
    answer = answer.strip().lower()
    return answer in ["yes", "y", "true", "1"]

def collect_facts():
    facts = dict(DEFAULT_FACTS)

    print("Medical Symptom Triage Bot")
    print("Answer the following questions.")
    print("-" * 40)

    facts["fever"] = ask_yes_no("Do you have fever?")
    facts["cough"] = ask_yes_no("Do you have cough?")
    facts["body_ache"] = ask_yes_no("Do you have body_ache?")
    facts["sore_throat"] = ask_yes_no("Do you have sore_throat?")
    facts["shortness_of_breath"] = ask_yes_no("Do you have shortness_of_breath?")
    facts["severe_chest_pain"] = ask_yes_no("Do you have severe_chest_pain?")

    return facts

def interactive_mode():
    facts = collect_facts()

    final_facts, fired_rules = forward_chain(facts, RULES, trace=True)

    print("\nConclusions:")
    for conclusion in get_conclusions(final_facts):
        print("-", conclusion)

    explain_inference(fired_rules, RULES)

In [24]:
interactive_mode()

Medical Symptom Triage Bot
Answer the following questions.
----------------------------------------
Do you have fever? (yes/no): yes
Do you have cough? (yes/no): yes
Do you have body_ache? (yes/no): yes
Do you have sore_throat? (yes/no): yes
Do you have shortness_of_breath? (yes/no): yes
Do you have severe_chest_pain? (yes/no): yes
Rule R7 fired.
Description: Severe chest pain requires emergency care.
New facts added: {'emergency': True}
--------------------------------------------------
Rule R1 fired.
Description: Fever, cough, and body ache suggest flu-like symptoms.
New facts added: {'flu_like': True}
--------------------------------------------------
Rule R2 fired.
Description: Fever and sore throat suggest possible throat infection.
New facts added: {'throat_infection_possible': True}
--------------------------------------------------
Rule R3 fired.
Description: Cough and shortness of breath require medical attention.
New facts added: {'respiratory_warning': True}
----------------

## Guide Questions and Answers

### 1. What is an expert system?
**Answer:** An expert system is a computer program that uses a knowledge base and a set of rules to solve problems or give advice in a specific domain, usually by mimicking how a human expert reasons.

### 2. What are the main components of an expert system?
**Answer:** An expert system has six main parts. The knowledge base holds the rules and facts, which in your notebook is the RULES list. The inference engine applies the rules to the facts, which is your forward_chain function. Working memory holds the current facts, which is your facts dictionary. The user interface lets users enter data and see results, which is your collect_facts and interactive_mode. The explanation facility shows how a conclusion was reached, which is your explain_inference. The knowledge acquisition facility lets experts add or edit rules, which is the RULES list itself.

### 3. What is forward chaining?
**Answer:** Forward chaining is reasoning from known facts to conclusions. It starts with what is known, checks which rules match, fires them, and adds new facts. It repeats until no more rules can fire. In your notebook, this is the forward_chain function.

### 4. How does your system use forward chaining?
**Answer:** My system uses forward chaining in the forward_chain function. It starts with the facts you enter, like fever or cough. Then it sorts the rules by priority and checks each one. If a rule's IF conditions match the current facts, the rule fires and its THEN facts are added to working memory. This repeats in a loop until no new facts are added. So it moves from symptoms to conclusions step by step.

### 5. Which rules fired in Test Case 1?
**Answer:** During Test Case 1, three rules triggered. R1 detected flu-like symptoms. R4 marked it as suspected flu. R5 advised rest and hydration.

### 6. Which rules fired in Test Case 2?
**Answer:** In Test Case 2, two rules fired. R3 raised a respiratory warning because of cough and shortness of breath. R6 then advised seeing a doctor.

### 7. Why is an explanation facility important?
**Answer:** The explain_inference function shows the IF-THEN logic and reason behind each fired rule. This gives transparency. Users can follow how a decision was made, and developers can check the logic and find bugs in the program.

### 8. What challenges did you encounter while designing the rules?
**Answer:** Some challenges were setting the right priority levels so urgent rules run first. Another was handling rule dependencies to avoid loops. It was also important to set correct boolean default facts so rules would not fire too early.

### 9. What are the limitations of your expert system?
**Answer:** The system only uses fixed and simple rules. It does not look at medical history, age, allergies, or how severe the symptoms are. It also cannot adapt or learn beyond the rules that were coded into it.



### 10. Why should this system not be used for real medical diagnosis?
**Answer:** That is because it is only an educational prototype for a computer science lab, not a certified clinical tool for hospitals. Real diagnosis needs professional clinical judgment, physical exams, and validated medical tests from trained professionals.